# FreeBSD Host Walkaround

Interactive architectural walkaround for the `freebsd-python-host` (FreeBSD Host (AI & Management)) runtime.


## 0. You Are Here
Execute an immediate runtime identity probe inside this kernel to verify the execution environment.


In [ ]:
%%sh
printf 'OS: '; uname -srm
printf 'User: '; whoami
printf 'Jailed: '; sysctl -n security.jail.jailed
printf 'VM Guest: '; sysctl -n kern.vm_guest
printf 'Hostname: '; hostname


## 1. Architecture: The Three Planes
The notebook document is hosted by JupyterLab on the host, while code cells execute in the selected runtime. Provisioning occurs on the host, outside guest execution environments.

```mermaid
graph LR
    Browser[JupyterLab / Notebook document]
    Server[Jupyter Server<br/>unprivileged host process]
    Provisioner[Kernel Provisioner]
    Daemon[Root runtime daemon]
    Runtime[Selected runtime]
    Kernel[ipykernel]

    Browser --> Server
    Server --> Provisioner
    Provisioner -->|Unix socket| Daemon
    Daemon --> Runtime
    Provisioner -->|SSH + port forwards| Runtime
    Runtime --> Kernel
```


## 2. Kernel Contract
**Security Boundary:** No additional FreeBSD Laboratory runtime-isolation boundary. Code executes as the Jupyter server OS user on the host and is therefore constrained only by normal host credentials, filesystem permissions, daemon authorization, and other host policy.

**Constraints & Requirements:**
- **Execution Plane:** Host environment (unprivileged Jupyter server user)
- **AI Inference:** Native in-process LLM inference with AVX2 SIMD acceleration
- **Control Plane:** Connects to root `runtime.sock` via `RuntimeClient` for orchestration
- **Capability Requirement:** `Host execution; in-process access to local GGUF models and AVX2 SIMD acceleration`


## 3. How It Is Launched
From [`freebsd_laboratory/kernels/freebsd-python-host/kernel.json`](../freebsd_laboratory/kernels/freebsd-python-host/kernel.json):

```json
{
  "argv": [
    "/home/freebsd/freebsd-laboratory/.venv/bin/python",
    "-m",
    "ipykernel_launcher",
    "-f",
    "{connection_file}"
  ],
  "display_name": "FreeBSD Host (AI & Management)",
  "language": "python",
  "interrupt_mode": "message",
  "metadata": {
    "debugger": true
  }
}
```

From [`freebsd_laboratory/ai.py`](../freebsd_laboratory/ai.py):

```python
def get_cached_llama(
    model_path: str | Path | None = None,
    n_ctx: int = 2048,
    n_gpu_layers: int = 0,
) -> Any:
    """Retrieve or initialize a cached Llama instance."""
    resolved = resolve_default_model(str(model_path) if model_path else None)
    if not resolved:
        raise RuntimeError("No GGUF model found. Specify model_path or place a model in /home/freebsd/models/")

    p = Path(resolved)
    if p.is_symlink() or not p.is_file():
        raise RuntimeError(f"Model path must be a regular file, not a symlink: {resolved}")

    cache_key = f"{resolved}:{n_ctx}:{n_gpu_layers}"
    if cache_key in _MODEL_CACHE:
```


## 4. Runtime Security Boundary
No additional FreeBSD Laboratory runtime-isolation boundary. Code executes as the Jupyter server OS user on the host and is therefore constrained only by normal host credentials, filesystem permissions, daemon authorization, and other host policy.

```mermaid
graph TD
    Browser[JupyterLab Browser] -->|NotebookActions / REST| Server[Jupyter Server (User: freebsd)]
    Server --> Kernel[Host ipykernel (.venv)]
    Kernel --> AI[llama-cpp-python / GGUF Models]
    Kernel -.->|Autonomous Tasks| Daemon[Root Runtime Daemon (via runtime.sock)]
    Daemon -.->|Provisions| Guests[Disposable Jails / bhyve VMs]
```


## 5. Inspect the Runtime
Execute safe, read-only shell observations inside this runtime.


In [ ]:
%%sh
uname -srm
sockstat -4 -l 2>/dev/null || netstat -tln 2>/dev/null
ifconfig 2>/dev/null || ip addr 2>/dev/null


## 6. Interpret the Evidence
Query the local in-process AI assistant directly using the `%ai` magic on the host:


In [ ]:
%load_ext freebsd_laboratory.magics
%ai Explain the architectural distinction between in-process host execution and disposable guest runtimes.

## 7. Bounded Investigation
Dispatch a bounded autonomous agent task from the host control plane to a fresh disposable guest runtime:


In [ ]:
%%agent --mode bhyve --steps 8
Launch a bhyve VM from the host control plane and verify its isolated network interfaces.

## 8. What You Cannot See
Explicitly distinguishing runtime-observable facts from control-plane facts:

- **Runtime-Observable Facts:** `uname`, active user identity, host network interfaces, process table, filesystem structure.
- **Control-Plane Facts:** Even on the host, Jupyter and this kernel execute as unprivileged user `freebsd`. Root-owned lifecycle operations (`zfs clone`, `jail -c`, `vm create`) still require delegating requests through `/var/run/freebsd-laboratory/runtime.sock`.


## 9. Summary & Design Invariants
Core architectural invariants:

1. **Document vs Runtime:** The notebook document is managed by JupyterLab on the host; code cells execute in the selected runtime.
2. **Transport Security:** Remote guest channels are tunneled exclusively over loopback SSH port forwards; the host kernel connects directly in-process.
3. **Privilege Separation:** The unprivileged Jupyter Server delegates privileged lifecycle operations to the root runtime daemon via `/var/run/freebsd-laboratory/runtime.sock`.
4. **Network Policy:** Guest environments cannot observe or alter host-side packet filtering (PF) policies.
5. **Ownership Scoping:** Destructive lifecycle operations (GC/cleanup) are strictly scoped to the authenticated owner's UID.
